In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# NLP library
import pandas as pd
import spacy

# Regular expression library
import re


In [2]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}



In [28]:
# current lemma exception dictionary for SpaCy
dict_lemma_lookup = {
    "international": "international",
    "internationale": "international",
    "Meister": "Meister",
    "kämpfend": "kämpfend",
    "abgebrüht": "abgebrüht",
    "beherrschend": "beherrschend",
    "entscheidend": "entscheidend",
    "entschlossen": "entschlossen",
    "angewiesen": "angewiesen",
    "berührt": "berührt",
    "besonnen": "besonnen",
    "betreut": "betreut",
    "bewegt": "bewegt",
    "einfühlend": "einfühlend",
    "engagiert": "engagiert",
    "entgegenkommend": "entgegenkommend",
    "ergreifend": "ergreifend",
    "fördernd": "fördernd",
    "gerührt": "gerührt",
    "heiter": "heiter",
    "lieb": "lieb",
    "mitfühlend": "mitfühlend",
    "mitwirkend": "mitwirkend",
    "motiviert": "motiviert",
    "nährend": "nährend",
    "teilnehmend": "teilnehmend",
    "unterstützend": "unterstützend",
    "verbindend": "verbindend",
    "vermittelnd": "vermittelnd",
    "vertraut": "vertraut",
    "weich": "weich",
    "zusammenhängend": "zusammenhängend",
    "zusammenwirkend": "zusammenwirkend",
    "zustimmend": "zustimmend",
    "jünger": "jünger",
    "ausgeprägt": "ausgeprägt",
    "ausgezeichnet": "ausgezeichnet",
    "äußerst": "äußerst",
    "beeindruckend": "beeindruckend",
    "beste": "beste",
    "bester": "beste",
    "besten": "beste",
    "bestem": "beste",
    "bestes": "beste",
    "etabliert": "etabliert",
    "führend": "führend",
    "fundiert": "fundiert",
    "gewandt": "gewandt",
    "Götter": "Götter",
    "hervorragend": "hervorragend",
    "überzeugend": "überzeugend",
    "zwingend": "zwingend",
}

In [4]:
# current SpaCy lookups table
lookup_table = model["de"].get_pipe("lemmatizer").lookups.get_table("lemma_lookup")


In [29]:
#Update of SpaCy lookups table with our custom distionary
for key in dict_lemma_lookup:
    lookup_table.set(key, dict_lemma_lookup[key])

In [31]:
# short test
model["de"]("hervorragend")[0].lemma_

'hervorragend'

## Generate lemmas from rules sheet to update existing dictionary 

In [32]:
# read rules with hyphens
df_rules = pd.read_csv("rules_16FEB_German_h.csv")


In [10]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 16 columns):
Lemma                             1790 non-null object
Word_Type                         1790 non-null object
Priority                          260 non-null object
Category                          1790 non-null object
Primary_subcategory               1790 non-null object
Secondary_subcategory             464 non-null object
Alt_Field                         799 non-null object
Alt_Sg_Replacement                580 non-null object
Alt_Sg_/_and_inclusive_form       281 non-null object
Alt_Sg_people_first               200 non-null object
Alt_Sg_reframed                   193 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            154 non-null object
Alt_Pl_pair_and_inclusive_form    245 non-null object
Alt_Pl_people_first               132 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

In [33]:
# remove all rows if lemmas is empty
df_rules = df_rules[df_rules['Lemma'].notna()]

In [12]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1790 entries, 0 to 1790
Data columns (total 16 columns):
Lemma                             1790 non-null object
Word_Type                         1790 non-null object
Priority                          260 non-null object
Category                          1790 non-null object
Primary_subcategory               1790 non-null object
Secondary_subcategory             464 non-null object
Alt_Field                         799 non-null object
Alt_Sg_Replacement                580 non-null object
Alt_Sg_/_and_inclusive_form       281 non-null object
Alt_Sg_people_first               200 non-null object
Alt_Sg_reframed                   193 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            154 non-null object
Alt_Pl_pair_and_inclusive_form    245 non-null object
Alt_Pl_people_first               132 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

## Split to words and sentences

In [13]:
list_word = []
list_sentence = []

list_word_type_word = []
list_word_type_sentence = []

list_priority_word = []
list_priority_sentence = []

list_category_word = []
list_category_sentence = []

list_primary_subcategory_word = []
list_primary_subcategory_sentence = []

for index, row in df_rules.iterrows():
    word = row["Lemma"]
    word_list = word.split() # Split `a_string` by whitespace.
    number_of_words = len(word_list)
    if number_of_words == 1:
        list_word.append(word)        
        list_word_type_word.append(row["Word_Type"])
        list_priority_word.append(row["Priority"])
        list_category_word.append(row["Category"])
        list_primary_subcategory_word.append(row["Primary_subcategory"])
        
        
    else:
        list_sentence.append(word)
        
        list_word_type_sentence.append(row["Word_Type"])
        list_priority_sentence.append(row["Priority"])
        list_category_sentence.append(row["Category"])
        list_primary_subcategory_sentence.append(row["Primary_subcategory"])
        

        
    


In [14]:
# dataframe with words
df_word = pd.DataFrame(list(zip(list_word,list_word_type_word, list_priority_word, list_category_word, list_primary_subcategory_word)), columns=["Lemma", "Word_Type", "Priority", "Category", "Primary_subcategory"])

In [15]:
# dataframe with sentences
df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_priority_sentence, list_category_sentence, list_primary_subcategory_sentence)), columns=["Lemma", "Word_Type", "Priority", "Category", "Primary_subcategory"])


In [16]:
#info about dataframe with words
df_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1616 entries, 0 to 1615
Data columns (total 5 columns):
Lemma                  1616 non-null object
Word_Type              1616 non-null object
Priority               192 non-null object
Category               1616 non-null object
Primary_subcategory    1616 non-null object
dtypes: object(5)
memory usage: 63.2+ KB


## Create Dictionaries

In [37]:
#dictionary with problematic SpaCy lemmas - go to human
words_lemma = {}

#dictionary with update lemmas 
words_dic = {}

for item in list_word:
    tokens = model["de"](item)
    for token in tokens:
        if item != token.lemma_:
            words_lemma[item]= token.lemma_
            words_dic[item]= item.strip()
        

In [38]:
# dictionary with problematic lemmas goes to human
words_lemma

{'Alter': 'alt',
 'Bucklige': 'bucklig',
 'Grundsätze': 'Grundsatz',
 'Herrschaften': 'Herrschaft',
 'Jeder': 'jed',
 'Kanus': 'Kanu',
 'Spitzenunternehmen ': 'Spitzenunternehmen',
 'Trampel': 'trampeln',
 'Wettkämpfe': 'Wettkampf',
 'Wilde': 'Wilder',
 'Zusammenhänge': 'Zusammenhang',
 'andauernd': 'andauern',
 'angreifend': 'angreifen',
 'anscheinend': 'anscheinen',
 'auffallend': 'auffallen',
 'aufstrebend': 'aufstreben',
 'ausgerechnet': 'ausrechnen',
 'bestimmend': 'bestimmen',
 'bestimmt': 'bestimmen',
 'einige': 'einig',
 'entschieden': 'entscheiden',
 'entspannt': 'entspannen',
 'erfüllend': 'erfüllen',
 'ermutigend': 'ermutigen',
 'erprobt': 'erproben',
 'etliche': 'etlich',
 'fortwährend': 'fortwähren',
 'führen': 'fahren',
 'gewagt': 'wagen',
 'herrschend': 'herrschen',
 'o.Ä.': 'Ä.',
 'offenbar': 'offenbaren',
 'schlicht': 'schleichen',
 'sicher': 'sichern',
 'stärker': 'stark',
 'treibend': 'treiben',
 'u.Ä.': 'Ä.',
 'u.ä.': '.',
 'zugegeben': 'zugeben',
 'überzeugt': 'übe

In [24]:
# dictionry with new lemmas
words_dic

{'Alter': 'Alter',
 'Bucklige': 'Bucklige',
 'Grundsätze': 'Grundsätze',
 'Herrschaften': 'Herrschaften',
 'Jeder': 'Jeder',
 'Kanus': 'Kanus',
 'Spitzenunternehmen ': 'Spitzenunternehmen',
 'Trampel': 'Trampel',
 'Wettkämpfe': 'Wettkämpfe',
 'Wilde': 'Wilde',
 'Zusammenhänge': 'Zusammenhänge',
 'andauernd': 'andauernd',
 'angreifend': 'angreifend',
 'anscheinend': 'anscheinend',
 'auffallend': 'auffallend',
 'aufstrebend': 'aufstrebend',
 'ausgerechnet': 'ausgerechnet',
 'bestimmend': 'bestimmend',
 'bestimmt': 'bestimmt',
 'einige': 'einige',
 'entschieden': 'entschieden',
 'entspannt': 'entspannt',
 'erfüllend': 'erfüllend',
 'ermutigend': 'ermutigend',
 'erprobt': 'erprobt',
 'etliche': 'etliche',
 'fortwährend': 'fortwährend',
 'führen': 'führen',
 'gewagt': 'gewagt',
 'herrschend': 'herrschend',
 'o.Ä.': 'o.Ä.',
 'offenbar': 'offenbar',
 'schlicht': 'schlicht',
 'sicher': 'sicher',
 'stärker': 'stärker',
 'treibend': 'treibend',
 'u.Ä.': 'u.Ä.',
 'u.ä.': 'u.ä.',
 'zugegeben': 'zu

## Update the lookup table

In [25]:
#new lookup table update 
dict_lemma_lookup.update(words_dic)

In [26]:
#new lookup table goes to model.py
dict_lemma_lookup

{'international': 'international',
 'internationale': 'international',
 'Meister': 'Meister',
 'kämpfend': 'kämpfend',
 'abgebrüht': 'abgebrüht',
 'beherrschend': 'beherrschend',
 'entscheidend': 'entscheidend',
 'entschlossen': 'entschlossen',
 'angewiesen': 'angewiesen',
 'berührt': 'berührt',
 'besonnen': 'besonnen',
 'betreut': 'betreut',
 'bewegt': 'bewegt',
 'einfühlend': 'einfühlend',
 'engagiert': 'engagiert',
 'entgegenkommend': 'entgegenkommend',
 'ergreifend': 'ergreifend',
 'fördernd': 'fördernd',
 'gerührt': 'gerührt',
 'heiter': 'heiter',
 'lieb': 'lieb',
 'mitfühlend': 'mitfühlend',
 'mitwirkend': 'mitwirkend',
 'motiviert': 'motiviert',
 'nährend': 'nährend',
 'teilnehmend': 'teilnehmend',
 'unterstützend': 'unterstützend',
 'verbindend': 'verbindend',
 'vermittelnd': 'vermittelnd',
 'vertraut': 'vertraut',
 'weich': 'weich',
 'zusammenhängend': 'zusammenhängend',
 'zusammenwirkend': 'zusammenwirkend',
 'zustimmend': 'zustimmend',
 'jünger': 'jünger',
 'ausgeprägt': 'au